In [1]:
import numpy as np
import pandas as pd
import json, os
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import torch, numpy as np
from sklearn.metrics import f1_score
from pathlib import Path

/Users/adiyeltay/mbzuai/nlp701_labs/semeval2025/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EMOTIONS = ["anger","fear","joy","sadness","surprise"]

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR.parent / "dataset"

train = pd.read_csv(DATA_DIR / "eng_train.csv")
dev = pd.read_csv(DATA_DIR / "eng_dev.csv")
test = pd.read_csv(DATA_DIR / "eng_test.csv")

def df_to_hf(df):
    return Dataset.from_pandas(df[["text"] + EMOTIONS], preserve_index=False)

hf_train = df_to_hf(train)
hf_dev = df_to_hf(dev)

In [3]:
MODEL_NAME = "xlm-roberta-base"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    enc = tok(batch["text"], truncation=True, padding="max_length", max_length=256)
    labels = np.stack([np.asarray(batch[e], dtype=np.float32) for e in EMOTIONS], axis=1)
    enc["labels"] = labels
    return enc

hf_train = hf_train.map(tokenize, batched=True, remove_columns=["text"]+EMOTIONS)
hf_dev = hf_dev.map(tokenize, batched=True, remove_columns=["text"]+EMOTIONS)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(EMOTIONS),
    problem_type="multi_label_classification"  # BCEWithLogitsLoss inside
)
collator = DataCollatorWithPadding(tokenizer=tok)

Map: 100%|██████████| 115/115 [00:00<00:00, 18837.19 examples/s]
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
from torch.utils.data import DataLoader
# batch = next(iter(DataLoader(hf_train, batch_size=4, collate_fn=collator)))
# print({k:(v.shape, v.dtype) for k,v in batch.items()})  # expect labels [4,6], float32

# model.eval()
# with torch.no_grad():
#     out = model(**{k:v.to(model.device) for k,v in batch.items()})
# print("forward loss:", float(out.loss))  # should be finite and > 0
# model.train()

import torch
model.eval()
batch = next(iter(DataLoader(hf_dev, batch_size=4, collate_fn=collator)))
with torch.no_grad():
    out = model(**{k:v.to(model.device) for k,v in batch.items()})
print("dev forward loss:", out.loss)

dev forward loss: tensor(0.6999)


In [5]:
def compute_metrics(p):
    logits = p.predictions
    if logits.ndim == 3:              # sometimes (N,1,C)
        logits = logits.squeeze(1)
    probs = 1/(1+np.exp(-logits))
    probs = np.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
    preds = (probs >= 0.5).astype(int)  # temp; thresholds later
    macro = f1_score(p.label_ids, preds, average="macro", zero_division=0)
    return {"macro_f1": macro}

args = TrainingArguments(
    output_dir="enc_emotions",
    learning_rate=1e-5, # 2
    max_grad_norm = 1.0,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    fp16=False, bf16=False, # MPS: keep off to avoid NaNs
    dataloader_pin_memory=False, # remove pin_memory warning
    remove_unused_columns=False, # don’t drop labels accidentally
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=hf_train,
    eval_dataset=hf_dev,
    processing_class=tok,                   # replaces deprecated tokenizer=
    data_collator=collator,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,0.553700,0.545109,0.181866
2,0.490800,0.486119,0.480188
3,0.443200,0.455564,0.538615
4,0.441100,0.448174,0.557003


TrainOutput(global_step=692, training_loss=0.4915871041358551, metrics={'train_runtime': 701.3849, 'train_samples_per_second': 15.763, 'train_steps_per_second': 0.987, 'total_flos': 1454517091540992.0, 'train_loss': 0.4915871041358551, 'epoch': 4.0})

In [6]:
# per label threshold tuning
dev_logits = trainer.predict(hf_dev).predictions
if dev_logits.ndim == 3:
    dev_logits = dev_logits.squeeze(1)
dev_probs = 1/(1+np.exp(-dev_logits))
dev_probs = np.nan_to_num(dev_probs, nan=0.0, posinf=1.0, neginf=0.0)
dev_true = np.stack([dev[e].values for e in EMOTIONS], axis=1).astype(int)

best_t = []
for j in range(len(EMOTIONS)):
    ts = np.linspace(0.05, 0.95, 37)
    f1s = [f1_score(dev_true[:,j], (dev_probs[:,j] >= t).astype(int)) for t in ts]
    best_t.append(float(ts[int(np.argmax(f1s))]))

macro = f1_score(dev_true, (dev_probs >= np.array(best_t)).astype(int), average="macro")
print("Best thresholds:", dict(zip(EMOTIONS, best_t)), "macro-F1:", macro)

os.makedirs("enc_emotions", exist_ok=True)
with open("enc_emotions/thresholds.json","w") as f:
    json.dump(dict(zip(EMOTIONS, best_t)), f, indent=2)

Best thresholds: {'anger': 0.15, 'fear': 0.25, 'joy': 0.44999999999999996, 'sadness': 0.475, 'surprise': 0.37499999999999994} macro-F1: 0.66811046298713
